In [4]:
avt_path = 'data/avt_tags.csv'
gidrootchistka_stabilizaciya_path = 'data/242000_tags.csv'
lims_path = 'ЛИМСы 01.01.2023 - н.в_ .xlsx'
pak_path = 'Выгрузка ПАК 01.01.2023 - н.в_.xlsx'
tags_path = 'Теги_хакатон.xlsx'

In [2]:
!pip install shap

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   ---------------------------------------- 2/2 [shap]



In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mlp
import seaborn as sns
import shap

In [5]:
avt_data = pd.read_csv(avt_path)
hydrostab_data = pd.read_csv(gidrootchistka_stabilizaciya_path)
lims = pd.read_excel(lims_path)
pak = pd.read_excel(pak_path)
la_tags = pd.read_excel(tags_path, sheet_name='ЛА')
vak_tags = pd.read_excel(tags_path, sheet_name='ВАК')
pak_tags = pd.read_excel(tags_path, sheet_name='ПАК')
kip_tags = pd.read_excel(tags_path, sheet_name='КИП')

In [6]:
lims.head()

,Установка 'АВТ'. Точка отбора '1'. Продукт 'Дизельное топливо',Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 98,Unnamed: 99,Unnamed: 100,Unnamed: 101,Unnamed: 102,Unnamed: 103,Unnamed: 104,Unnamed: 105,Unnamed: 106,Unnamed: 107
0,CFPP,NaN,90%.T,NaN,50%.T,NaN,EBP.T,NaN,PourPoint,NaN,...,PourPoint,NaN,50%.T,NaN,CetaneNumber,NaN,90%.T,NaN,I350,NaN
1,°С,NaN,°С,NaN,кг/м3,NaN,% об.,NaN,°С,NaN,...,°С,NaN,°С,NaN,ед.цет.ч.,NaN,°С,NaN,% об.,NaN
2,Количество значений:,44.0,Количество значений:,580.0,Количество значений:,648.0,Количество значений:,648.0,Количество значений:,1255.0,...,Количество значений:,33.0,Количество значений:,1252.0,Количество значений:,42.000000,Количество значений:,1252.0,Количество значений:,1279.0
3,2025-10-17 08:00:00,1.0,2023-01-02 14:00:00,344.0,2023-01-02 14:00:00,305.0,2023-01-02 14:00:00,370.0,2023-01-01 14:00:00,-5.0,...,2023-03-12 10:00:00,-13.0,2023-01-01 10:00:00,271.0,2023-01-11 10:00:00,56.299999,2023-01-01 10:00:00,325.0,2023-01-01 10:00:00,97.0
4,2025-10-20 06:00:41,2.0,2023-01-04 14:00:00,346.0,2023-01-04 14:00:00,307.0,2023-01-04 14:00:00,370.0,2023-01-02 14:00:00,-7.0,...,2023-04-19 09:30:00,-8.0,2023-01-02 10:00:00,273.0,2023-02-01 10:00:00,55.000000,2023-01-02 10:00:00,327.0,2023-01-02 10:00:00,97.0


In [38]:
avt_data = avt_data.drop(columns=avt_data.columns[range(2)])
avt_data.transpose()



,0,1,2,3,4,5,6,7,8,9,...,189207,189208,189209,189210,189211,189212,189213,189214,189215,189216
date,2023-01-01 00:00:00,2023-01-01 00:10:00,2023-01-01 00:20:00,2023-01-01 00:30:00,2023-01-01 00:40:00,2023-01-01 00:50:00,2023-01-01 01:00:00,2023-01-01 01:10:00,2023-01-01 01:20:00,2023-01-01 01:30:00,...,2026-08-06 22:30:00,2026-08-06 22:40:00,2026-08-06 22:50:00,2026-08-06 23:00:00,2026-08-06 23:10:00,2026-08-06 23:20:00,2026-08-06 23:30:00,2026-08-06 23:40:00,2026-08-06 23:50:00,2026-08-07 00:00:00
T1,130.372528,130.312668,130.254074,130.195465,130.131912,130.068039,130.018433,130.149307,130.280182,130.397247,...,131.159286,131.350174,131.11467,131.386871,131.189011,131.194336,131.336182,131.114349,131.180481,131.112106
P2,3.540594,3.535826,3.532611,3.532775,3.53294,3.534257,3.537143,3.540029,3.542386,3.544277,...,3.859151,3.849393,3.840634,3.838976,3.837317,3.83644,3.837928,3.839417,3.837819,3.830853
F3,68.645889,68.768883,68.994743,68.718155,68.386543,69.145332,69.722099,69.440147,69.40757,69.980949,...,107.093132,107.153114,107.086044,106.852806,107.119125,107.416397,107.527336,107.628471,107.398705,107.154549
P4,3.848177,3.847023,3.84483,3.841944,3.839058,3.844698,3.85385,3.863002,3.863137,3.861246,...,4.180276,4.170184,4.162111,4.161716,4.16132,4.161318,4.162167,4.163015,4.160222,4.152978
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
P67,1.099674,1.101833,1.101986,1.101378,1.10077,1.100258,1.099756,1.099232,1.098286,1.097341,...,1.098499,1.095056,1.091612,1.093438,1.095849,1.09826,1.101448,1.104692,1.107407,1.102721
F68,28.852953,30.252674,29.917879,28.748285,28.555887,29.643618,29.790293,28.57406,28.730316,29.68144,...,12.634389,12.444997,14.359883,14.758203,16.39999,14.538986,14.293461,13.952545,14.576883,16.046112
F69,71.61824,75.51162,75.045563,69.521233,70.300598,72.964081,73.138855,69.487633,70.869179,74.233643,...,42.177486,38.688362,49.062378,46.063564,50.914875,45.043201,44.930656,43.773613,45.973644,51.154568
W70,77.383003,77.023788,76.893791,77.121658,77.33812,77.312294,77.287445,77.282623,77.308289,77.737122,...,114.62767,114.889824,114.972679,114.72477,114.9785,114.78476,115.044792,114.850548,114.850952,114.562569


In [44]:
hydrostab_data = hydrostab_data.drop(columns=hydrostab_data.columns[range(1)])

In [45]:
hst_trans = hydrostab_data.transpose()



In [46]:
hst_trans.head()

,0,1,2,3,4,5,6,7,8,9,...,189207,189208,189209,189210,189211,189212,189213,189214,189215,189216
date,2023-01-01 00:00:00,2023-01-01 00:10:00,2023-01-01 00:20:00,2023-01-01 00:30:00,2023-01-01 00:40:00,2023-01-01 00:50:00,2023-01-01 01:00:00,2023-01-01 01:10:00,2023-01-01 01:20:00,2023-01-01 01:30:00,...,2026-08-06 22:30:00,2026-08-06 22:40:00,2026-08-06 22:50:00,2026-08-06 23:00:00,2026-08-06 23:10:00,2026-08-06 23:20:00,2026-08-06 23:30:00,2026-08-06 23:40:00,2026-08-06 23:50:00,2026-08-07 00:00:00
F1,2.613271,2.551895,2.660722,2.542066,2.597317,2.5382,2.508729,2.536158,2.605648,2.598691,...,5.17379,5.144533,5.333936,5.324295,5.322152,5.329049,5.479344,5.499983,5.405285,5.410972
F2,88476.804688,88497.3125,88954.195312,88769.585938,88832.539062,88482.671875,88058.125,87892.75,87924.546875,87598.671875,...,95238.429688,95136.546875,95083.90625,95476.546875,95346.335938,95301.578125,95329.054688,95346.375,95449.609375,95707.617188
P3,3.580906,3.603907,3.62392,3.632347,3.634601,3.626778,3.615397,3.603061,3.590726,3.576519,...,3.648319,3.635745,3.629131,3.638496,3.647051,3.655331,3.66361,3.65851,3.65144,3.642647
W4,1.817558,1.824941,1.82157,1.819784,1.819716,1.811073,1.793564,1.797447,1.82258,1.84754,...,3.826004,3.869102,3.926512,3.931377,3.929414,3.97385,4.023729,4.01462,3.998835,3.936888


In [6]:
kip_tags.head()

,АВТ (описание),АВТ,24-2000 (описание),24-2000
0,Температура верха К1. Выход на FIRC0955,T1,Расход бензина c установки,F1
1,Давление бензиновых паров верха К1,P2,Газовая схема. Расход на линии от ЦК-201,F2
2,Расход бензина на орошение К1,F3,Полисеп. Сепаратор С-201. Давление на входе,P3
3,Давление низа К1 в Е6,P4,Блок стабилизации. Массовый расход бензина в к...,W4
4,Расход пара в К1,F5,Полисеп. Реактор Р-201. Температура ГСС на выходе,T5


In [8]:
kip_avt_tags = pd.read_excel(tags_path, sheet_name='КИП', usecols='A:B')
kip_242000_tags = pd.read_excel(tags_path, sheet_name='КИП', usecols='C:D')

In [9]:
kip_avt_tags.head()

,АВТ (описание),АВТ
0,Температура верха К1. Выход на FIRC0955,T1
1,Давление бензиновых паров верха К1,P2
2,Расход бензина на орошение К1,F3
3,Давление низа К1 в Е6,P4
4,Расход пара в К1,F5


In [10]:
kip_242000_tags.head()

,24-2000 (описание),24-2000
0,Расход бензина c установки,F1
1,Газовая схема. Расход на линии от ЦК-201,F2
2,Полисеп. Сепаратор С-201. Давление на входе,P3
3,Блок стабилизации. Массовый расход бензина в к...,W4
4,Полисеп. Реактор Р-201. Температура ГСС на выходе,T5


In [18]:
#Подключаю имена из тэгов
hs_data = pd.merge(hydrostab_data, kip_242000_tags, on='')

KeyError: ''

In [7]:
lims.head()


,Установка 'АВТ'. Точка отбора '1'. Продукт 'Дизельное топливо',Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 98,Unnamed: 99,Unnamed: 100,Unnamed: 101,Unnamed: 102,Unnamed: 103,Unnamed: 104,Unnamed: 105,Unnamed: 106,Unnamed: 107
0,CFPP,NaN,90%.T,NaN,50%.T,NaN,EBP.T,NaN,PourPoint,NaN,...,PourPoint,NaN,50%.T,NaN,CetaneNumber,NaN,90%.T,NaN,I350,NaN
1,°С,NaN,°С,NaN,кг/м3,NaN,% об.,NaN,°С,NaN,...,°С,NaN,°С,NaN,ед.цет.ч.,NaN,°С,NaN,% об.,NaN
2,Количество значений:,44.0,Количество значений:,580.0,Количество значений:,648.0,Количество значений:,648.0,Количество значений:,1255.0,...,Количество значений:,33.0,Количество значений:,1252.0,Количество значений:,42.000000,Количество значений:,1252.0,Количество значений:,1279.0
3,2025-10-17 08:00:00,1.0,2023-01-02 14:00:00,344.0,2023-01-02 14:00:00,305.0,2023-01-02 14:00:00,370.0,2023-01-01 14:00:00,-5.0,...,2023-03-12 10:00:00,-13.0,2023-01-01 10:00:00,271.0,2023-01-11 10:00:00,56.299999,2023-01-01 10:00:00,325.0,2023-01-01 10:00:00,97.0
4,2025-10-20 06:00:41,2.0,2023-01-04 14:00:00,346.0,2023-01-04 14:00:00,307.0,2023-01-04 14:00:00,370.0,2023-01-02 14:00:00,-7.0,...,2023-04-19 09:30:00,-8.0,2023-01-02 10:00:00,273.0,2023-02-01 10:00:00,55.000000,2023-01-02 10:00:00,327.0,2023-01-02 10:00:00,97.0


In [ ]:
pak.drop(0, inplace=True)
pak.drop
pak.head()



,24-2000:Mg.Sulfur,Unnamed: 1,Unnamed: 2,24-2000:D15,Unnamed: 4
1,2023-01-01 00:00:00,6.572924,NaN,2025-03-05 10:20:00,831.5055
2,2023-01-01 00:10:00,6.730094,NaN,2025-03-05 10:30:00,831.3279
3,2023-01-01 00:20:00,7.158879,NaN,2025-03-05 10:40:00,831.4442
4,2023-01-01 00:30:00,7.620472,NaN,2025-03-05 10:50:00,831.2587
5,2023-01-01 00:40:00,8.173620,NaN,2025-03-05 11:00:00,831.1998


In [9]:
tags.head()

,Установка 'АВТ'. Точка отбора '1'. Продукт 'ФРАКЦ_ДИЗ',Установка 'АВТ'. Точка отбора '2'. Продукт 'Дизельное топливо',Установка 'АВТ'. Точка отбора '2.1'. Продукт 'Дизельное топливо',Установка 'АВТ'. Точка отбора '3'. Продукт 'Дизельное топливо',Установка 'Гидроочистка'. Точка отбора '1'. Продукт 'ФРАКЦ_ДИЗ'.,Установка 'Гидроочистка'.. Точка отбора '2'. Продукт 'Дизельное топливо'
0,ПТФ,Т50%,Температура помутнения,Т90%,ПТФ,Температура помутнения
1,Т90%,Конец кипения,Конец кипения,Т50%,Т90%,Плотность при 15 °С
2,Т50%,Среднее плотность,Среднее плотность,ПТФ,Начало кипения,Температура вспышки
3,Температура помутнения до 350,Температура помутнения,Т50%,Т95%,Т50%,ТНК
4,Конец кипения,Начало кипения,Начало кипения,Начало кипения,Т95%,ОДИС 250°С


In [16]:
pak_sulphur = pak[0,1]


KeyError: (0, 1)